In [ ]:
from pathlib import Path
import torch
import re

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
except Exception as e:
    tokenizer = None
    print(f"Tokenizer load failed: {e}")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
    ).to(device)
    model.eval()
except Exception as e:
    model = None
    print(f"Model load failed: {e}")


def generate_text(prompt, max_length=1000, num_return_sequences=1):
    if tokenizer is None or model is None:
        return "Model could not be loaded."
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

In [ ]:
database_path = Path("database")
documents = []

for md_file in database_path.glob("*.md"):
    loader = TextLoader(str(md_file), encoding="utf-8")
    documents.extend(loader.load())